# Getting Started with OpenRouter and LLMs

**A beginner-friendly, hands-on tutorial** for calling a large language model (LLM) from Python using OpenRouter and the OpenAI-compatible API.

In this notebook, you will:
- understand what OpenRouter does;
- prepare a Python environment;
- store an API key safely in a `.env` file;
- send a prompt to an LLM;
- classify a social-media comment as positive, negative, or neutral;
- experiment with prompts and troubleshoot common errors.

> **Important:** API requests can incur charges. Check the model's pricing and your OpenRouter account limits before running the request.

## 1. What is OpenRouter?

OpenRouter provides a unified API for accessing models from different AI providers. Instead of integrating each provider separately, you can use an OpenAI-compatible client and set its `base_url` to OpenRouter.

The general flow is:

1. Your Python notebook creates messages containing instructions and user input.
2. The OpenAI Python SDK sends the request to OpenRouter.
3. OpenRouter routes the request to the selected model.
4. The model's response is returned to your notebook.

A model name must be a model identifier supported by OpenRouter. The example below uses `openai/gpt-4o-mini` as an illustrative model slug; check the OpenRouter model catalogue and your account's availability/pricing before use.

## 2. Prerequisites

Before starting, make sure you have:

- **Python 3.10 or newer** installed.
- **Cursor or Jupyter Notebook** with the Python and Jupyter extensions (for Cursor).
- A project virtual environment, preferably managed with `uv`.
- An **OpenRouter account** and an API key created through the OpenRouter dashboard.
- Internet access from the machine running this notebook.

### Install the required Python packages

Run this in the terminal from your project directory. If you manage dependencies with `uv`, use:

```bash
uv sync
```

Select your project's `.venv/bin/python` as the notebook kernel. If you use plain `pip` instead, activate your virtual environment first and install `openai`, `python-dotenv`, and `ipykernel` there.

**Why these packages?**
- `openai`: provides the OpenAI-compatible Python client used to call OpenRouter.
- `python-dotenv`: reads environment variables from a local `.env` file.
- `ipykernel`: allows the notebook to execute code using the selected Python environment.

## 3. Configure your API key safely

Create a file named **`.env`** in the same project directory as this notebook (or in the working directory from which you run it). Add your key like this:

```dotenv
OPENROUTER_API_KEY=your_openrouter_api_key_here
```

Replace the placeholder with your actual key. Do not add the key to a notebook cell, commit it to Git, post it in a screenshot, or share it in a public repository.


If you accidentally expose a key, revoke it in your OpenRouter account and create a replacement.

## 4. Load and validate the API key

The next cell loads the `.env` file and checks that a key is present. OpenRouter keys commonly begin with `sk-or-`, but a prefix check is only a basic sanity check—it does not prove that the key is valid. Never print the full key.

In [1]:
import os
from dotenv import load_dotenv

# Load variables from .env and override existing environment variables.
# Use override=True only if you intentionally want .env values to take precedence.
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("No API key found. Check that .env exists and contains OPENROUTER_API_KEY.")
elif api_key.strip() != api_key:
    print("The API key has leading or trailing whitespace. Remove it from .env.")
elif not api_key.startswith("sk-or-"):
    print("The key does not have the expected OpenRouter prefix. Check that you copied an OpenRouter API key.")
else:
    print("API key is present and its format looks plausible. The key has not yet been authenticated.")

API key is present and its format looks plausible. The key has not yet been authenticated.


### If the key check fails

- Confirm `.env` is in the notebook's current working directory, or load it using an explicit path.
- Check that the variable is spelled exactly `OPENROUTER_API_KEY`.
- Remove quotes or whitespace if you copied them accidentally. A typical `.env` line is `OPENROUTER_API_KEY=sk-or-...`.
- Do not paste your API key into chat or debugging output.

You can inspect the current working directory without revealing secrets:

In [ ]:
from pathlib import Path
print("Notebook working directory:", Path.cwd())
print(".env exists here:", (Path.cwd() / ".env").exists())

## 5. Create the OpenRouter client

The OpenAI Python SDK can be configured to use OpenRouter by setting `base_url`. The client reads the key from the environment variable.

**Model choice:** Model slugs, availability, capabilities, and pricing change. Verify the model identifier in OpenRouter's model catalogue. The model below is an example; replace it with a model available to your account if necessary.

In [2]:
from openai import OpenAI

if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing. Complete the API-key setup section first.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

# Example OpenRouter model slug. Confirm it is currently available before running.
MODEL = "openai/gpt-4o-mini"

print("OpenRouter client created. No request has been sent yet.")

OpenRouter client created. No request has been sent yet.


## 6. Understand system prompts and user prompts

A chat request contains a list of messages with roles:

- **`system`**: sets the assistant's behavior and constraints.
- **`user`**: supplies the task or content to process.
- **`assistant`**: represents a previous model response, when maintaining a conversation.

For this example, the system prompt asks the model to classify a social-media comment using exactly one label. The user prompt should contain the actual comment to analyze; weather information is not itself a comment, so we use a realistic sample comment.

In [3]:
system_prompt = """
You are an expert comment analyzer.
Classify the given social-media comment as exactly one of these labels:
positive, negative, or neutral.
Return only the single label, with no explanation or punctuation.
""".strip()

user_comment = "The delivery was quick and the product works perfectly. I'm very happy!"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Classify this comment:\n\n{user_comment}"},
]

print("System prompt:\n", system_prompt)
print("\nComment to classify:\n", user_comment)

System prompt:
 You are an expert comment analyzer.
Classify the given social-media comment as exactly one of these labels:
positive, negative, or neutral.
Return only the single label, with no explanation or punctuation.

Comment to classify:
 The delivery was quick and the product works perfectly. I'm very happy!


## 7. Send the request to the LLM

This cell sends a real API request. It may use account credits or incur charges depending on the selected model and your account settings. Keep the comment non-sensitive for this community exercise.

The `reasoning_effort` parameter from some model-specific examples is intentionally omitted here because support for it varies by model and API route. Start with the broadly supported chat-completions arguments, then consult the selected model's documentation for optional parameters.

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
    max_tokens=10,
)

result = response.choices[0].message.content
print("Predicted sentiment:", result)

Predicted sentiment: positive


## 8. Try different comments

Change the comment below and rerun the cell to observe how the result changes. This is a simple demonstration, not a perfect sentiment-analysis system: sarcasm, mixed opinions, slang, and missing context can lead to mistakes.

In [ ]:
def classify_comment(comment: str) -> str:
    """Send one comment to the configured model and return its text response."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Classify this comment:\n\n{comment}"},
        ],
        temperature=0,
        max_tokens=10,
    )
    return (response.choices[0].message.content or "").strip()

examples = [
    "This is the best service I've used. Friendly staff and fast delivery!",
    "The app crashes every time I try to pay. Very frustrating.",
    "The store opens at 9 AM and is located near the station.",
]

for comment in examples:
    print(f"Comment: {comment}")
    print(f"Sentiment: {classify_comment(comment)}")
    print()

## 9. Experiment with the prompt

Try these experiments:

1. Add an instruction to treat mixed praise and criticism as `neutral`.
2. Ask for JSON output such as `{"sentiment": "positive"}`. Then validate the returned JSON in Python.
3. Test a sarcastic comment, such as: `Great, another update that broke everything.`
4. Compare two different models, checking both quality and cost.

Prompt wording can influence results. For reliable applications, create a representative test set, measure errors, and validate outputs rather than assuming every model response is correct.

## 10. Warnings and good practices

- **Protect API keys:** never hard-code secrets in notebooks or commit `.env` to version control. Notebook outputs and shared files can accidentally reveal sensitive values.
- **Watch costs:** requests can incur charges. Check pricing, account limits, and usage before running loops or batch jobs.
- **Check model identifiers:** a model slug may be unavailable, renamed, restricted, or incompatible with an optional parameter.
- **Treat outputs as fallible:** an LLM can produce an incorrect label or violate the requested format. Validate responses before using them in a real system.
- **Avoid sensitive data:** do not send private, regulated, or personally identifying information unless you have confirmed the applicable policies and permissions.
- **Be cautious with untrusted text:** user comments can contain misleading instructions. In production systems, keep the classification instructions separate from the content being classified and validate the output.
- **Understand provider routing and policies:** review OpenRouter's current privacy, data-handling, and provider-routing information for the model you choose.
- **Don't share raw errors containing secrets:** redact keys, headers, and private data before sharing logs.

## 11. Troubleshooting

| Symptom | Likely cause | What to check |
|---|---|---|
| `ModuleNotFoundError: No module named 'dotenv'` | Package missing from the notebook's active kernel | Run `uv add python-dotenv` in the project and select `.venv/bin/python` as the kernel. |
| `ModuleNotFoundError: No module named 'openai'` | OpenAI SDK missing from active environment | Run `uv add openai` and verify the notebook kernel. |
| API key missing | `.env` not found or variable misspelled | Check `Path.cwd()`, `.env` location, and `OPENROUTER_API_KEY`. |
| Authentication error / HTTP 401 | Key invalid, revoked, or copied incorrectly | Create or copy a valid OpenRouter key; do not print it. |
| Model not found / HTTP 404 | Wrong or unavailable model slug | Copy the exact model identifier from the OpenRouter catalogue. |
| Rate limit / HTTP 429 | Usage limit or rate limit reached | Check account limits, credits, and retry policy. |
| Connection / timeout error | Network or service issue | Check connectivity and OpenRouter status, then retry cautiously. |

To verify the notebook interpreter, run:

```python
import sys
print(sys.executable)
```

It should point to your project environment, for example `.../llm_garage/.venv/bin/python`.

## 12. Summary

You have now seen the basic workflow for connecting to an LLM through OpenRouter:

1. Install packages in the project's Python environment.
2. Store the API key in `.env` and keep it private.
3. Configure the OpenAI client with OpenRouter's API base URL.
4. Send system and user messages to a supported model.
5. Read and validate the response.

Next, you can explore structured outputs, streaming, conversation history, and model comparisons.